# Model V1

In [ ]:
import sys
sys.path.append('../data_extraction')

import os
import numpy as np
import pandas as pd
import mlflow
import keras
import matplotlib.pyplot as plt
import tensorflow as tf

import mlflow.keras
from keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, Concatenate, Dropout
from keras.regularizers import l2

from crater_extraction import template_match_t, match_coords, filter_to_detectable, filter_edge_craters, truth_coords_for_patch
from LRO_data_class import getSplitIndices, getNormalisedBatch, patchGenerator, stepsPerEpoch, percentileNormalise

# paths and label tables - every cell below reads these
PATCHES_DIR = '../pre_processing/lunar_patches'
LABELS_CSV = '../data_preparation/filtered_labels.csv'

kept_labels = pd.read_csv(os.path.join(PATCHES_DIR, 'kept_labels.csv'))
filtered_labels = pd.read_csv(LABELS_CSV)

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [9]:
train_idx, val_idx, test_idx = getSplitIndices()
print(f'train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')

# sanity check on one file - training reads through patchGenerator, not from here
wac, dem, mask = getNormalisedBatch(batch_num=0)

print(f'wac  {wac.shape}  {wac.dtype}   [{wac.min():.3f}, {wac.max():.3f}]')
print(f'dem  {dem.shape}  {dem.dtype}   [{dem.min():.3f}, {dem.max():.3f}]')
print(f'mask {mask.shape}  {mask.dtype}   crater pixels {mask.mean()*100:.2f}%')


train: 136109  val: 34699  test: 35059
(1000, 256, 256) float32 0.0 1.0


## Hyperparameters

In [10]:
# all hyperparameters in one place — change here and MLflow logs them automatically

# one place to change the seed - generators and weight init both read it
SEED = 42

# set_random_seed covers python/numpy/tf seeds but NOT cuDNN kernel choice.
# enable_op_determinism makes GPU ops deterministic, at a speed cost.
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

params = {
    'dim': 256,
    'channels': 'both',                 # 'both' | 'wac' | 'dem'
    'input_channels': 2,                # 2 for both, 1 for ablations
    'n_filters': 32,                    # DeepMoon used 112 (paper 2.3)
    'FL': 3,                            # kernel size
    'init': 'he_normal',
    'lmbda': 1e-6,                      # L2. NB paper 2.7 says 1e-5, repo says 1e-6 - sources disagree
    'dropout': 0.15,
    'learning_rate': 0.0001,
    'batch_size': 8,
    'epochs': 20,
    'loss': 'binary_focal_crossentropy',
    'focal_alpha': 0.75,                # weight on class 1, the rim. rare at 37:1 so it takes the larger share
    'focal_gamma': 2.0,
    'focal_class_balancing': True,
    'model': 'U-Net-v1',
    'seed': SEED,                        # same batch order + augmentation across all runs
}

## Model Architecture

In [11]:
img_input = keras.Input(shape=(params['dim'], params['dim'], params['input_channels']))

# Encoder1
a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(img_input)

a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1)

a1P = MaxPooling2D((2, 2), strides=(2, 2))(a1)

# Encoder2
a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1P)

a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2)

a2P = MaxPooling2D((2, 2), strides=(2, 2))(a2)

# Encoder3
a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2P)

a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3)

a3P = MaxPooling2D((2, 2), strides=(2, 2))(a3)

# Encoder4
a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3P)

a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4)

a4P = MaxPooling2D((2, 2), strides=(2, 2))(a4)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4P)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(u)

# Decoder1
d1CT = Conv2DTranspose(params['n_filters']*8, kernel_size=2, strides=2, padding='same')(u)
d1c = Concatenate()([d1CT, a4])
x1 = Dropout(params['dropout'])(d1c)

d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x1)

d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d1)

# Decoder2
d2CT = Conv2DTranspose(params['n_filters']*4, kernel_size=2, strides=2, padding='same')(d1)
d2c = Concatenate()([d2CT, a3])
x2 = Dropout(params['dropout'])(d2c)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x2)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d2)

# Decoder3
d3CT = Conv2DTranspose(params['n_filters']*2, kernel_size=2, strides=2, padding='same')(d2)
d3c = Concatenate()([d3CT, a2])
x3 = Dropout(params['dropout'])(d3c)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x3)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d3)

# Decoder4
d4CT = Conv2DTranspose(params['n_filters'], kernel_size=2, strides=2, padding='same')(d3)
d4c = Concatenate()([d4CT, a1])
x4 = Dropout(params['dropout'])(d4c)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x4)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d4)

# Output layer
output = Conv2D(1, 1, activation='sigmoid')(d4)
model = keras.Model(img_input, output)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 256, 256,  │        608 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 256, 256,  │      9,248 │ conv2d_19[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 128, 128,  │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 128, 128,  │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_21[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 64, 64,    │          0 │ conv2d_22[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_23[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 32, 32,    │          0 │ conv2d_24[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_6[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 32, 32,    │    590,080 │ conv2d_25[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 16, 16,    │          0 │ conv2d_26[0][0]   │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_7[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 16, 16,    │  2,359,808 │ conv2d_27[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 32, 32,    │    524,544 │ conv2d_28[0][0]   │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 32, 32,    │          0 │ conv2d_transpose

 Total params: 7,759,809 (29.60 MB)

 Trainable params: 7,759,809 (29.60 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# loss is constructed from params so the two can't drift apart
# MLflow logs the params
loss_fn = keras.losses.BinaryFocalCrossentropy(
    apply_class_balancing=params['focal_class_balancing'],
    alpha=params['focal_alpha'],
    gamma=params['focal_gamma'],
)

model.compile(optimizer=keras.optimizers.Adam(params['learning_rate']), loss=loss_fn)

# EarlyStopping: stop once val_loss stops improving for `patience` epochs and roll
# back to the best weights - without restore_best_weights you keep the overfit ones.
os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1, min_delta=1e-4),
    keras.callbacks.ModelCheckpoint(
        f"checkpoints/{params['model']}_{params['channels']}_{params['n_filters']}f_s{params['seed']}_{{epoch:02d}}.keras",
        monitor='val_loss', save_best_only=True, verbose=1),
]


In [13]:
# SMOKE TEST - not a real run. 8 patches, no augmentation, no callbacks.
# Loss must fall toward 0 - 8 patches repeated is trivially memorisable.
# If it plateaus, the bug is upstream (mask alignment, channel order), not the model.
# model.fit(
#     patchGenerator(train_idx[:8], channels=params['channels'],
#                    batch_size=params['batch_size'], augment_data=False),
#     steps_per_epoch=10,
#     epochs=params['epochs'],
# )


## Training

In [14]:
# MLflow tracks every training run — hyperparameters, metrics, and the model itself
# run `mlflow ui` in the terminal and localhost:5000
# each run is logged separately to compare experiments side by side

# [source]: https://mlflow.org/docs/latest/python_api/mlflow.keras.html
# [example source]: https://github.com/mlflow/mlflow/blob/master/examples/keras/train.py

mlflow.set_experiment('lunar-crater-detection')

with mlflow.start_run(run_name=f"{params['channels']}-{params['n_filters']}f-s{params['seed']}") as run:
    mlflow.log_params(params)

    history = model.fit(
        patchGenerator(train_idx, channels=params['channels'], batch_size=params['batch_size'],
                       rng=np.random.default_rng(params['seed'])),
        steps_per_epoch=stepsPerEpoch(train_idx, params['batch_size']),
        validation_data=patchGenerator(val_idx, channels=params['channels'], batch_size=params['batch_size'],
                                       augment_data=False, rng=np.random.default_rng(params['seed'])),
        validation_steps=stepsPerEpoch(val_idx, params['batch_size']),
        epochs=params['epochs'],
        callbacks=callbacks,
    )

    for epoch, (tl, vl) in enumerate(zip(history.history['loss'], history.history['val_loss'])):
        mlflow.log_metric('train_loss', tl, step=epoch)
        mlflow.log_metric('val_loss', vl, step=epoch)

    mlflow.keras.log_model(model, 'model')

    run_id = run.info.run_id

/home/ppxsv1/miniconda3/envs/lunar_lro/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a generator. The generator is expected to yield already-shuffled data.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/20
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 0.0099
Epoch 1: val_loss improved from None to 0.00944, saving model to checkpoints/U-Net-v1_both_01.keras

Epoch 1: finished saving model to checkpoints/U-Net-v1_both_01.keras
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 2902s 170ms/step - loss: 0.0099 - val_loss: 0.0094
Epoch 2/20
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 0.0082
Epoch 2: val_loss improved from 0.00944 to 0.00908, saving model to checkpoints/U-Net-v1_both_02.keras

Epoch 2: finished saving model to checkpoints/U-Net-v1_both_02.keras
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 2944s 173ms/step - loss: 0.0082 - val_loss: 0.0091
Epoch 3/20
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 0.0079
Epoch 3: val_loss did not improve from 0.00908
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 2768s 163ms/step - loss: 0.0079 - val_loss: 0.0091
Epoch 4/20
17013/17013 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 0.0078
Epoch 4: val_loss improved from 0.00908 to 0.00887, saving model to 

2026/08/15 16:15:48 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/08/15 16:16:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


## Evaluation

Pixel-space, per patch. Extraction and matching live in `crater_extraction.py` so the baseline and this model run the identical pipeline.

Comparable to DeepMoon's *post-CNN* column (57% recall), not their *post-processed* 92% - that figure comes from merging detections across ~120 overlapping views per crater, which this single-patch setup does not do.


In [ ]:
# filtered_labels.csv holds only Robbins columns - wac_col/wac_row are added in
# data_pre_processing after the csv is saved, so they survive only in kept_labels.
# lat/lon -> pixel is linear, so the mapping is recovered by fitting the crater
# rows of kept_labels. no hardcoded tile shape, so this still works per tile
# when the dataset moves to all 8.
crater_rows = kept_labels.dropna(subset=['LON_CIRC_IMG', 'wac_col'])

col_fit = np.polyfit(crater_rows['LON_CIRC_IMG'], crater_rows['wac_col'], 1)
row_fit = np.polyfit(crater_rows['LAT_CIRC_IMG'], crater_rows['wac_row'], 1)

all_wac_col = np.polyval(col_fit, filtered_labels['LON_CIRC_IMG'].values)
all_wac_row = np.polyval(row_fit, filtered_labels['LAT_CIRC_IMG'].values)
all_diameters = filtered_labels['DIAM_CIRC_IMG'].values

print(f'lon -> col: {col_fit[0]:.2f} px/deg     lat -> row: {row_fit[0]:.2f} px/deg')


In [ ]:
# Threshold sweep
# target_thresh inherited from DeepMoon at 0.1, where the loss was unweighted BCE.
# focal squashes the output into a narrow low band, so 0.1 can binarise the whole
# patch into one blob and match nothing. both tails go to zero - notes 17.3
# runs on VALIDATION - tuning a threshold on test contaminates every number after it
# same truth source as the headline metric, or it optimises the wrong target

thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7]

n_sweep = 200

# own generator - this runs before the evaluation, which seeds its own
sweep_rng = np.random.default_rng(params['seed'])
sweep_idx = np.sort(sweep_rng.choice(val_idx, size=n_sweep, replace=False))

# predictions and truth are the same at every threshold, so build them once
sweep_pred = []
sweep_truth = []

loaded_file = -1

for patch_idx in sweep_idx:

    file_num = int(patch_idx // 1000)
    position = patch_idx % 1000

    if file_num != loaded_file:
        raw_wac = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
        raw_dem = np.load(os.path.join(PATCHES_DIR, f'X_dem_{file_num}.npz'))['arr_0']
        loaded_file = file_num

    wac_patch = percentileNormalise(raw_wac[position])
    dem_patch = percentileNormalise(raw_dem[position])

    if params['channels'] == 'both':
        patch_input = np.stack([wac_patch, dem_patch], axis=-1)
    elif params['channels'] == 'wac':
        patch_input = wac_patch[..., None]
    else:
        patch_input = dem_patch[..., None]

    sweep_pred.append(model.predict(patch_input[None, ...], verbose=0)[0, :, :, 0])

    row = kept_labels.iloc[patch_idx]
    truth = truth_coords_for_patch(row['center_col'], row['center_row'], row['patch_lat'],
                                   all_wac_col, all_wac_row, all_diameters)
    sweep_truth.append(filter_edge_craters(filter_to_detectable(truth)))

sweep_precision = []
sweep_recall = []
sweep_f1 = []

for threshold in thresholds:

    swept_match = 0
    swept_detected = 0
    swept_truth = 0

    for prediction, truth in zip(sweep_pred, sweep_truth):

        detections = filter_edge_craters(template_match_t(prediction.copy(), target_thresh=threshold))

        match_count, detection_count, truth_count, _, _, _ = match_coords(truth, detections)

        swept_match += match_count
        swept_detected += detection_count
        swept_truth += truth_count

    if swept_detected > 0:
        sweep_precision.append(swept_match / swept_detected)
    else:
        sweep_precision.append(0)

    if swept_truth > 0:
        sweep_recall.append(swept_match / swept_truth)
    else:
        sweep_recall.append(0)

    if sweep_precision[-1] + sweep_recall[-1] > 0:
        sweep_f1.append(2 * sweep_precision[-1] * sweep_recall[-1] / (sweep_precision[-1] + sweep_recall[-1]))
    else:
        sweep_f1.append(0)

    print(f'threshold {threshold}: P {sweep_precision[-1]:.3f}  R {sweep_recall[-1]:.3f}  F1 {sweep_f1[-1]:.3f}')

# the value to carry into evaluation. the optimum is interior, so it is read off here
# rather than assumed - a flat zero row means the grid missed it, widen and re-run
best_threshold = thresholds[int(np.argmax(sweep_f1))]

print(f'best threshold: {best_threshold}   F1 {max(sweep_f1):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(thresholds, sweep_precision, marker='o', label='precision')
axes[0].plot(thresholds, sweep_recall, marker='o', label='recall')
axes[0].plot(thresholds, sweep_f1, marker='o', label='F1')
axes[0].set_xlabel('target_thresh')
axes[0].legend()

axes[1].plot(sweep_recall, sweep_precision, marker='o')
axes[1].set_xlabel('recall')
axes[1].set_ylabel('precision')

plt.show()


In [ ]:
# one batch of test patches to look at
eval_gen = patchGenerator(test_idx, channels=params['channels'], batch_size=params['batch_size'],
                          augment_data=False, rng=np.random.default_rng(params['seed']))

X, y = next(eval_gen)

pred = model.predict(X)

fig, axes = plt.subplots(1, 4)

# one patch - change i to look at another in the batch
i = 0
axes[0].imshow(X[i,:,:,0], cmap='gray')
axes[0].set_title('WAC')

axes[1].imshow(X[i,:,:,1], cmap='terrain')
axes[1].set_title('DEM')

axes[2].imshow(y[i,:,:,0], vmin=0, vmax=1)
axes[2].set_title('True Mask')

axes[3].imshow(pred[i,:,:,0], vmin=0, vmax=1)
axes[3].set_title(f'Pred: max{pred[i].max():.3f}')

plt.show()

In [ ]:
coords = template_match_t(pred[i, :, :, 0].copy(), target_thresh=best_threshold)

print('Detections: ', len(coords))

fig, ax = plt.subplots()
ax.imshow(X[i, :, :, 0], cmap='gray')

for x, yy, r in coords:
    ax.add_patch(plt.Circle((x, yy), r, fill=False, color='red'))

plt.show()

In [ ]:
all_coords = []

for p in range(len(pred)):
    
    c = template_match_t(pred[p, :, :, 0].copy(), target_thresh=best_threshold)
    all_coords.append(c)

    print('Detections (patch ', p,'): ', len(c))

In [ ]:
crater_detections = template_match_t(pred[i, :, :, 0].copy(), target_thresh=best_threshold)
ground_truth  = filter_to_detectable(template_match_t(y[i, :, :, 0].copy()))

print('Craters in true mask: ', len(ground_truth))
print('Detected:', len(crater_detections))

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

for a, (title, coords) in zip(ax, [('true', ground_truth), ('pred', crater_detections)]):
        
    a.imshow(X[i, :, :, 0], cmap='gray')

    for x, yy, r in coords:
        a.add_patch(plt.Circle((x, yy), r, fill=False, color='red'))
    
    a.set_title(f'{title}: {len(coords)}')

plt.show()

In [ ]:
# For visualisations and checks only - 1 patch
match_count, detection_count, truth_count, matched_pairs, false_positives, multi_match_count = match_coords(ground_truth, crater_detections)

if detection_count > 0:
    precision = match_count / detection_count
else:
    precision = 0

if truth_count > 0:
    recall = match_count / truth_count
else:
    recall = 0

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0

print(f'TP: {match_count}   detected: {detection_count}   truth: {truth_count}')
print(f'P: {precision:.3f}   R: {recall:.3f}   F1: {f1:.3f}')
print(f'detections claiming >1 truth crater: {multi_match_count}')


In [ ]:
# All patches
# iterates test_idx directly, not the generator - the generator shuffles and
# never says which patch it handed back, so its patches cannot be looked up
# in kept_labels. also lets the sample be random rather than whatever the
# file-walk produced.
# runs at best_threshold from the sweep above, not the inherited default

n_eval = 160

rng = np.random.default_rng(params['seed'])
# sorted so patches from the same npz are consecutive and the file can be reused
eval_idx = np.sort(rng.choice(test_idx, size=n_eval, replace=False))

total_match = 0
total_detected = 0
total_truth = 0

all_matched_pairs = []
all_false_positives = []
all_truth_radii = []
total_multi_match = 0

# getNormalisedBatch normalises all 1000 patches in a file to serve one, so it is
# loaded raw here and only the wanted patch is normalised
loaded_file = -1

for patch_idx in eval_idx:

    file_num = int(patch_idx // 1000)
    position = patch_idx % 1000

    if file_num != loaded_file:
        raw_wac = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
        raw_dem = np.load(os.path.join(PATCHES_DIR, f'X_dem_{file_num}.npz'))['arr_0']
        loaded_file = file_num

    wac_patch = percentileNormalise(raw_wac[position])
    dem_patch = percentileNormalise(raw_dem[position])

    if params['channels'] == 'both':
        patch_input = np.stack([wac_patch, dem_patch], axis=-1)
    elif params['channels'] == 'wac':
        patch_input = wac_patch[..., None]
    else:
        patch_input = dem_patch[..., None]

    prediction = model.predict(patch_input[None, ...], verbose=0)

    detections = filter_edge_craters(template_match_t(prediction[0, :, :, 0].copy(), target_thresh=best_threshold))

    row = kept_labels.iloc[patch_idx]
    truth = truth_coords_for_patch(row['center_col'], row['center_row'], row['patch_lat'],
                                   all_wac_col, all_wac_row, all_diameters)
    truth = filter_to_detectable(truth)
    truth = filter_edge_craters(truth)

    match_count, detection_count, truth_count, matched_pairs, false_positives, multi_match_count = match_coords(truth, detections)

    total_match += match_count
    total_detected += detection_count
    total_truth += truth_count
    total_multi_match += multi_match_count

    if len(matched_pairs) > 0:
        all_matched_pairs.append(matched_pairs)

    if len(false_positives) > 0:
        all_false_positives.append(false_positives)

    if len(truth) > 0:
        all_truth_radii.append(truth[:, 2])

if len(all_matched_pairs) > 0:
    all_matched_pairs = np.vstack(all_matched_pairs)
else:
    all_matched_pairs = np.empty((0, 6))

if len(all_false_positives) > 0:
    all_false_positives = np.vstack(all_false_positives)
else:
    all_false_positives = np.empty((0, 3))

if len(all_truth_radii) > 0:
    all_truth_radii = np.concatenate(all_truth_radii)
else:
    all_truth_radii = np.empty(0)

print(f'TP: {total_match}   detected: {total_detected}   truth: {total_truth}')
print(f'detections claiming >1 truth crater: {total_multi_match}')


In [ ]:
# Overall crater-level metrics
if total_detected > 0:
    precision = total_match / total_detected
else:
    precision = 0

if total_truth > 0:
    recall = total_match / total_truth
else:
    recall = 0

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0

print(f'P: {precision:.3f}   R: {recall:.3f}   F1: {f1:.3f}')

# logged to the training run so the six runs can be compared in one place
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('n_eval_patches', n_eval)
    mlflow.log_metric('multi_match', total_multi_match)
    mlflow.log_metric('target_thresh', best_threshold)


In [ ]:
# Loss curves

best_epoch = int(np.argmin(history.history['val_loss']))

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.axvline(best_epoch, color='grey', linestyle='--', label=f'best epoch ({best_epoch + 1})')

plt.xlabel('epoch')
plt.ylabel(params['loss'])
plt.legend()
plt.show()


In [ ]:
# Recall by crater diameter
# diameter_km = 2 * r_px * 0.1 -> bins 1-2 / 2-5 / 5-10 km are r = 5 / 10 / 25 / 50
# DeepMoon: recall drops above r = 15 px, 3 km here. notes 13.1

bin_edges = [5, 10, 25, 50]
bin_labels = ['1-2 km', '2-5 km', '5-10 km']

matched_truth_radii = all_matched_pairs[:, 5]

bin_recall = []

for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):

    truth_in_bin = ((all_truth_radii >= lower) & (all_truth_radii < upper)).sum()
    matched_in_bin = ((matched_truth_radii >= lower) & (matched_truth_radii < upper)).sum()

    if truth_in_bin > 0:
        bin_recall.append(matched_in_bin / truth_in_bin)
    else:
        bin_recall.append(0)

    print(f'{lower}-{upper} px: {matched_in_bin} / {truth_in_bin}')

plt.bar(bin_labels, bin_recall)
plt.ylabel('recall')
plt.ylim(0, 1)
plt.title('Recall by crater diameter')
plt.show()


In [ ]:
# Crater size-frequency distribution
# parallel to the catalogue -> extra detections behave like real craters

all_detected_radii = np.concatenate([all_matched_pairs[:, 2], all_false_positives[:, 2]])

detected_diameters = all_detected_radii * 2 * 0.1
truth_diameters = all_truth_radii * 2 * 0.1

diameter_bins = np.logspace(np.log10(1), np.log10(10), 15)

plt.hist(truth_diameters, bins=diameter_bins, histtype='step', label='Robbins (in patch)')
plt.hist(detected_diameters, bins=diameter_bins, histtype='step', label='detected')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('diameter (km)')
plt.ylabel('count')
plt.legend()
plt.show()


In [ ]:
# Positional and radius error
# fractional, over the mean radius. DeepMoon medians <= 11% (Table 3.1)

mean_radius = (all_matched_pairs[:, 2] + all_matched_pairs[:, 5]) / 2

error_x = abs(all_matched_pairs[:, 0] - all_matched_pairs[:, 3]) / mean_radius
error_y = abs(all_matched_pairs[:, 1] - all_matched_pairs[:, 4]) / mean_radius
error_radius = abs(all_matched_pairs[:, 2] - all_matched_pairs[:, 5]) / mean_radius

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, values, name in zip(axes, [error_x, error_y, error_radius], ['x', 'y', 'radius']):

    ax.hist(values, bins=40)
    ax.axvline(np.median(values), color='red', linestyle='--')
    ax.set_title(f'{name}: median {np.median(values):.3f}')
    ax.set_xlabel('fractional error')

plt.show()


In [ ]:
# False positives
# Robbins incomplete near 1 km - some of these are real, uncatalogued

n_show = 12

n_show = min(n_show, len(all_false_positives))
sample = all_false_positives[rng.choice(len(all_false_positives), n_show, replace=False)]

print('false positives: ', len(all_false_positives))
print(sample)
